##Portfolio insights & Report Automation

Steps:
1. Import libraries
2. Import raw data
3. Inspect the data
4. Enrich the data

 • derive calculations

 • get stock market data

5. EDA (Exporatory Data Analysis)
6. Data prep for reporting
7. Embed GenAI
Building the report

##Step 1 and 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from datetime import datetime, timedelta
import yfinance as yf



In [3]:
input_file = "portfolio_data.csv"
output_file = "portfolio_data_final.csv"

In [4]:
raw_df = pd.read_csv(input_file)

In [5]:
raw_df.sample(10)

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price
1612,PF-MAT-0163,VMC,Vulcan Materials Company,371,Materials,160.92
603,PF-CON-0056,COST,Costco Wholesale Corp.,210,Consumer Goods,1163.95
1288,PF-IND-0123,FDX,FedEx Corporation,345,Industrials,1353.89
329,PF-REA-0035,EQR,Equity Residential,412,Real Estate,1401.17
755,PF-ENE-0069,SLB,SLB (Schlumberger),315,Energy,804.88
352,PF-ENE-0027,CVX,Chevron Corporation,340,Energy,1418.34
314,PF-REA-0030,CCI,Crown Castle Inc.,306,Real Estate,1188.20
1938,PF-HEA-0171,LLY,Eli Lilly and Company,45,Healthcare,1495.26
1414,PF-TEC-0176,AAPL,Apple Inc.,25,Technology,585.05
203,PF-MAT-0028,NEM,Newmont Corporation,266,Materials,702.18


In [6]:
print("Count of rows and columns:",raw_df.shape)

Count of rows and columns: (2000, 6)


In [7]:
raw_df.isna().sum()

,0
portfolio_id,0
stock_symbol,0
stock_name,0
quantity,0
stock_sector,0
purchase_price,0


In [8]:
null_count = int(raw_df.isna().sum().sum())

print(f"We have {null_count} nulls (missing values) in this dataset.")

We have 0 nulls (missing values) in this dataset.


In [9]:
dup_cont = raw_df.duplicated().sum()

print(f"We have {dup_cont} duplicates in this dataset.")

We have 4 duplicates in this dataset.


In [10]:
#Don't Run
#drop duplicates and define a new dataset called clean_df
clean_df = raw_df.drop_duplicates(subset=['purchase_price']) #intentional to show how mistakes can be found
clean_df.head()

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price
0,PF-REA-0001,DLR,Digital Realty Trust,58,Real Estate,47.27
1,PF-ENE-0001,XOM,Exxon Mobil Corporation,126,Energy,342.58
2,PF-MAT-0001,IP,International Paper Co.,53,Materials,1018.28
3,PF-UTI-0001,AEP,American Electric Power,45,Utilities,889.83
4,PF-TEC-0001,NVDA,NVIDIA Corporation,16,Technology,149.61


In [11]:
clean_df.shape

(1984, 6)

In [12]:
clean_df = raw_df.drop_duplicates()
clean_df.shape

(1996, 6)

We lost a huge volume of data.
We had 2000 rows, with 4 duplicates gone,we get 1996 rows

##Step 4

In [13]:
#Extract unique ticket- make sure null tickers are dropped(just in case)
tickers = clean_df['stock_symbol'].dropna().unique().tolist()
tickers

['DLR',
 'XOM',
 'IP',
 'AEP',
 'NVDA',
 'MS',
 'SRE',
 'NEM',
 'HON',
 'AAPL',
 'MRK',
 'OXY',
 'CSCO',
 'CCI',
 'MLM',
 'WMT',
 'O',
 'KO',
 'CRM',
 'NFLX',
 'BAC',
 'SHW',
 'AXP',
 'CB',
 'PCG',
 'WFC',
 'WELL',
 'UNH',
 'D',
 'TGT',
 'VZ',
 'COST',
 'LMT',
 'EXC',
 'CAT',
 'VMC',
 'FDX',
 'IBM',
 'SO',
 'MPC',
 'PARA',
 'C',
 'INTC',
 'FOX',
 'EQR',
 'TMO',
 'JPM',
 'LIN',
 'DUK',
 'GS',
 'AMT',
 'NEE',
 'GE',
 'WEC',
 'RTX',
 'PG',
 'BMY',
 'ADBE',
 'AMD',
 'DE',
 'ORCL',
 'CHTR',
 'PXD',
 'META',
 'SPGI',
 'UPS',
 'PSX',
 'BA',
 'SLB',
 'NUE',
 'TSLA',
 'MSFT',
 'ECL',
 'TMUS',
 'SPG',
 'WBD',
 'SBUX',
 'VLO',
 'AMGN',
 'APD',
 'AMZN',
 'FCX',
 'EOG',
 'NKE',
 'PFE',
 'MDT',
 'CVX',
 'JNJ',
 'HD',
 'ES',
 'CMCSA',
 'MCD',
 'BLK',
 'GOOGL',
 'DIS',
 'QCOM',
 'PLD',
 'AVB',
 'EQIX',
 'T',
 'ABT',
 'MMM',
 'COP',
 'PEP',
 'LLY']

In [14]:
len(tickers)

105

Because we to extract, current_price,
1,3,5- years performence, we need to define the timeframes.

In [15]:
current_year = datetime.now().year
ytd_start = pd.Timestamp(f"{current_year}-01-01")
ytd_start

Timestamp('2026-01-01 00:00:00')

In [16]:
three_years_ago = datetime.now() - timedelta(days=3*365)
five_years_ago = datetime.now() - timedelta(days=5*365)

print(three_years_ago, five_years_ago)

2023-06-13 00:37:31.937991 2021-06-13 00:37:31.938052


In [17]:
prices = yf.download(tickers, start=five_years_ago.strftime('%Y-%m-%d')
                     ,auto_adjust=True, progress=False)['Close']

ERROR:yfinance:HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PARA"}}}
ERROR:yfinance:
2 Failed downloads:
ERROR:yfinance:['PARA', 'PXD']: YFTzMissingError('possibly delisted; no timezone found')


In [18]:
prices

Ticker,AAPL,ABT,ADBE,AEP,AMD,AMGN,AMT,AMZN,APD,AVB,...,UPS,VLO,VMC,VZ,WBD,WEC,WELL,WFC,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2021-06-14,127.185638,100.574127,556.950012,71.065041,81.550003,206.053070,232.838898,169.193497,264.006866,180.574356,...,160.429352,69.470215,165.012726,41.922997,30.820000,78.272285,71.796684,40.027912,43.956394,51.646416
2021-06-15,126.366837,100.510422,548.460022,71.156853,80.470001,204.908310,232.434967,169.156494,265.736725,177.843140,...,161.426575,69.766922,165.329834,42.003635,30.070000,78.618530,71.752495,40.302746,43.781265,53.526886
2021-06-16,126.863998,100.191788,543.330017,70.455635,80.110001,204.677628,229.049026,170.762497,264.818909,176.764130,...,159.894791,69.054825,163.715515,41.527069,30.459999,77.630516,71.434227,40.400269,42.889996,53.335514
2021-06-17,128.462540,101.621010,551.359985,70.447296,84.559998,205.685745,232.718887,174.462006,263.980438,177.042267,...,158.003998,66.613426,159.554886,41.446419,29.410000,78.010529,71.213226,37.935650,43.068253,51.579861
2021-06-18,127.166145,100.437584,565.590027,68.744354,84.650002,203.908737,229.636169,174.345001,258.340515,173.737778,...,157.780640,64.443268,159.353073,40.925869,29.080000,75.519363,69.409668,37.013634,42.270805,50.256874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-06-05,307.339996,91.070000,251.440002,129.139999,466.380005,349.579987,194.119995,246.029999,282.350006,189.720001,...,108.540001,255.820007,281.380005,45.369999,26.240000,112.949997,206.929993,81.940002,118.879997,149.919998
2026-06-08,301.540009,90.500000,244.990005,126.769997,490.329987,345.730011,189.100006,245.220001,276.769989,187.610001,...,107.699997,258.390015,269.980011,45.439999,26.469999,111.250000,200.000000,80.959999,119.830002,151.750000
2026-06-09,290.549988,91.250000,237.880005,127.760002,475.510010,344.570007,190.830002,244.190002,282.980011,186.850006,...,107.870003,253.779999,279.000000,45.779999,26.559999,113.099998,206.770004,82.000000,118.879997,148.910004


In [19]:
#Convert a single ticker Series into a DataFrame so processing stays consistent
if isinstance(prices, pd.Series):
    prices = prices.to_frame(name=prices.name) # Use prices.name for the column name

# Ensure tickers is correctly defined before looping
tickers = clean_df['stock_symbol'].dropna().unique().tolist()

# Srore calculated market metrics for ticker
market_data = {}

# Loop throught each stack ticker to calculate performance metrics
for ticker in tickers:
  try:
     # Remove missing price values for cleaner calculations
     s = prices[ticker].dropna() # Corrected: use 'ticker' instead of 'tickers' for column selection

     # Stop processing if no historical price data exists
     if s.empty:
        raise ValueError(f"No historical price data available for {ticker}") # Added f-string for better error message

     # Get the most recent closing price
     current_price = s.iloc[-1]

     # Calculate percentage return from a given start date
     def perf_since(start_date):
          period = s.loc[start_date:]

          # Return None if no date exists after the selected date
          if period.empty:
             return None

          # Compute percentage growth from start date to current price
          return ((current_price / period.iloc[0]) -1) * 100 # Corrected: 'price' to 'period'

     # Save calculated metrics for the current ticker
     market_data[ticker] = { # Corrected: 'tickers' to 'ticker'
         'current_price': round(current_price, 2),
         'ytd_Perf': round(perf_since(ytd_start) or 0, 2),
         '3YR_Perf': round(perf_since(three_years_ago) or 0, 2),
         '5YR_Perf': round(perf_since(five_years_ago) or 0, 2), # Consistent use of perf_since
     }

  # Use fallback values if API data retrieval fails
  except Exception as e:
     # print(f"Warning: Failed to process ticker {ticker}. Using fallback values. Error: {e}") # Optional: for debugging
     market_data[ticker] = { # Corrected: 'tickers' to 'ticker'
         "current_price": 0.0, # Placeholder as stock_master_data is undefined
         'ytd_Perf': 5.0,
         '3YR_Perf': 15.0,
         '5YR_Perf': 45.0,
     }

In [20]:
market_data

{'DLR': {'current_price': np.float64(182.84),
  'ytd_Perf': np.float64(18.74),
  '3YR_Perf': np.float64(88.99),
  '5YR_Perf': np.float64(32.71)},
 'XOM': {'current_price': np.float64(146.6),
  'ytd_Perf': np.float64(21.14),
  '3YR_Perf': np.float64(54.11),
  '5YR_Perf': np.float64(183.85)},
 'IP': {'current_price': np.float64(34.95),
  'ytd_Perf': np.float64(-10.99),
  '3YR_Perf': np.float64(25.74),
  '5YR_Perf': np.float64(-26.02)},
 'AEP': {'current_price': np.float64(128.48),
  'ytd_Perf': np.float64(12.63),
  '3YR_Perf': np.float64(71.7),
  '5YR_Perf': np.float64(80.79)},
 'NVDA': {'current_price': np.float64(204.87),
  'ytd_Perf': np.float64(8.62),
  '3YR_Perf': np.float64(377.4),
  '5YR_Perf': np.float64(1040.91)},
 'MS': {'current_price': np.float64(212.66),
  'ytd_Perf': np.float64(18.19),
  '3YR_Perf': np.float64(165.96),
  '5YR_Perf': np.float64(175.38)},
 'SRE': {'current_price': np.float64(91.54),
  'ytd_Perf': np.float64(2.75),
  '3YR_Perf': np.float64(36.71),
  '5YR_Perf'

In [21]:
market_data_df = pd.DataFrame(market_data).T
market_data_df.head(10)

,current_price,ytd_Perf,3YR_Perf,5YR_Perf
DLR,182.84,18.74,88.99,32.71
XOM,146.60,21.14,54.11,183.85
IP,34.95,-10.99,25.74,-26.02
AEP,128.48,12.63,71.70,80.79
NVDA,204.87,8.62,377.40,1040.91
MS,212.66,18.19,165.96,175.38
SRE,91.54,2.75,36.71,49.64
NEM,97.59,-3.17,147.64,61.56
HON,219.12,13.04,23.66,15.57
AAPL,295.63,9.29,62.97,132.44


> Now that we have our marketdata extracted,let's implement it into our original main dataframe(using pandas)

> we have unique tickers with their data. The original data contains portfolios where the ticker is repeated. Therefore, we need to map the stock data into the dataframe.

In [22]:
clean_df = clean_df.copy()
clean_df.loc[:, 'current_price'] = clean_df['stock_symbol'].map(lambda x: market_data[x]['current_price'])
clean_df.loc[:, 'ytd_%'] = clean_df['stock_symbol'].map(lambda x: market_data[x]['ytd_Perf'])
clean_df.loc[:, '3YR_%'] = clean_df['stock_symbol'].map(lambda x: market_data[x]['3YR_Perf'])
clean_df.loc[:, '5YR_%'] = clean_df['stock_symbol'].map(lambda x: market_data[x]['5YR_Perf'])

In [23]:
#Values and costs
clean_df['total_cost'] = clean_df['purchase_price'] * clean_df['quantity']
clean_df['current_value'] = clean_df['current_price'] * clean_df['quantity']

In [24]:
#percent of pertfolios
#let's inpsect the outcome of our grop by (data aggregation)
portfolios_totals_sum = clean_df.groupby('portfolio_id')['current_value'].sum()
portfolios_totals_sum

,current_value
portfolio_id,
PF-COM-0001,12109.23
PF-COM-0002,32345.46
PF-COM-0003,1173.50
PF-COM-0004,0.00
PF-COM-0005,2208.96
...,...
PF-UTI-0207,15603.66
PF-UTI-0208,63083.68
PF-UTI-0209,50316.15


In [25]:
#now we can the calculation in a fraction
portfolios_totals = clean_df.groupby('portfolio_id')['current_value'].transform('sum')
clean_df['percent_of_portfolio'] = round((clean_df['current_value'] / portfolios_totals) * 100, 2)

In [26]:
clean_df.sample(10)

,portfolio_id,stock_symbol,stock_name,quantity,stock_sector,purchase_price,current_price,ytd_%,3YR_%,5YR_%,total_cost,current_value,percent_of_portfolio
1192,PF-CON-0123,PG,Procter & Gamble Co.,436,Consumer Goods,104.36,148.34,6.16,9.51,24.88,45500.96,64676.24,100.0
793,PF-TEC-0097,META,Meta Platforms Inc.,206,Technology,86.22,568.43,-12.53,109.58,70.11,17761.32,117096.58,100.0
828,PF-MAT-0080,MLM,Martin Marietta Materials,32,Materials,1061.61,565.54,-10.62,35.10,67.91,33971.52,18097.28,100.0
1014,PF-MAT-0100,NEM,Newmont Corporation,116,Materials,456.96,97.59,-3.17,147.64,61.56,53007.36,11320.44,100.0
371,PF-CON-0031,HD,The Home Depot Inc.,428,Consumer Goods,728.92,326.01,-4.39,17.37,19.36,311977.76,139532.28,100.0
1692,PF-ENE-0166,MPC,Marathon Petroleum Corp.,259,Energy,1323.27,260.81,59.33,148.43,366.20,342726.93,67549.79,100.0
113,PF-ENE-0006,PXD,Pioneer Natural Resources,479,Energy,398.54,0.00,5.00,15.00,45.00,190900.66,0.00,NaN
712,PF-TEC-0085,NVDA,NVIDIA Corporation,65,Technology,110.18,204.87,8.62,377.40,1040.91,7161.70,13316.55,100.0
1363,PF-TEC-0169,AMZN,Amazon.com Inc.,459,Technology,1217.93,241.51,6.63,91.04,42.74,559029.87,110853.09,100.0
736,PF-IND-0080,MMM,3M Company,14,Industrials,547.74,157.91,-1.47,103.28,12.10,7668.36,2210.74,100.0


Let's find a `portfolio_id` that contains multiple stocks to demonstrate the `percent_of_portfolio` calculation.

In [27]:
# Aggregate the date
stock_allocation = clean_df.groupby('stock_name').agg(current_allocation = ('current_value', 'sum')).reset_index()


In [28]:
stock_allocation

,stock_name,current_allocation
0,3M Company,868820.82
1,AT&T Inc.,69207.00
2,Abbott Laboratories,370882.05
3,Adobe Inc.,1041925.60
4,Advanced Micro Devices,2329906.50
...,...,...
100,WEC Energy Group,614083.17
101,Walmart Inc.,773971.50
102,Warner Bros. Discovery,144533.66
103,Wells Fargo & Company,542274.40


In [29]:
# Calculate the allocation percentage
total_current_value_stocks = stock_allocation['current_allocation'].sum()
stock_allocation['allocation_%'] = round((stock_allocation['current_allocation'] / total_current_value_stocks) * 100, 2)


#sort for better visualization
stock_allocation = stock_allocation.sort_values(by='allocation_%',ascending=False)
stock_allocation

,stock_name,current_allocation,allocation_%
44,Goldman Sachs Group,8072813.80,7.05
34,Eli Lilly and Company,4373298.65,3.82
35,Equinix Inc.,3975558.98,3.47
26,Costco Wholesale Corp.,3399303.96,2.97
58,Meta Platforms Inc.,3383295.36,2.95
...,...,...,...
72,Pfizer Inc.,115959.27,0.10
69,PG&E Corporation,108094.02,0.09
1,AT&T Inc.,69207.00,0.06
70,Paramount Global,0.00,0.00


In [30]:
# Sanity check - all percentages should add up to 100
stock_allocation['allocation_%'].sum()

np.float64(99.89000000000001)

In [31]:
fig = go.Figure(data=[go.Pie(labels= stock_allocation.sample(10)
['stock_name'], values=stock_allocation['allocation_%'],hole = 0.3)]) #Set the hole size for the donut chart

fig.update_layout(title_text='Portfolio Allocation by Stack Name')
fig.show()

In [32]:
import plotly.graph_objects as go

fig_col = go.Figure(data=[go.Bar(x=stock_allocation['stock_name'], y=stock_allocation['current_allocation'])])

fig_col.update_layout(
    title='Portfolio Current Stock Values by Stock Name',
    xaxis_title='Stock Name',
    yaxis_title='Current Stock Value',
    xaxis={'categoryorder':'total descending'}
)

fig_col.show()